# COMP5329 — Deep Learning
**Week 8 · Self-Study Notebook — Sequence Modeling Architectures II**

**Semester 1, 2026**

---

### What this notebook is

This is the single authoritative reference for Week 8. It covers:

1. **Every topic from the lecture** (Quick Review · BERT · GPT · Vision Transformer · Mamba/SSM), in more depth than the slides.
2. **Every topic from the old tutorial notebook** — the 60-minute live tutorial now focuses on the SSM duality only; everything else (GPT/BERT depth, full Mamba internals, RWKV, KAN) lives here.
3. **Out-of-lecture extensions** clearly labelled as such — RWKV and KAN are included as Appendix A and B for students who want alternative routes beyond the Mamba line the lecture takes.

### How to read it

- §§ 1–5 track the lecture directly. Read them in order.
- § 5 (Efficient sequence modelling / SSM) is the most technically dense section; it is also the subject of the live tutorial, so some material overlaps deliberately — use the duplication to reinforce the recurrent ↔ convolutional duality.
- § 6 is a grand comparison table across all models covered.
- **Appendix A (RWKV)** and **Appendix B (KAN)** are self-contained; you can read them in any order, or skip them.

### What you should have from Week 7

- Scaled dot-product attention: $\text{softmax}(QK^\top/\sqrt d)V$ and its $O(T^2)$ cost.
- Vanilla RNN recurrence and the vanishing/exploding gradient problem.
- LSTM cell state and gating.
- Transformer encoder/decoder block structure (attention → residual → FFN → residual).

If any of that is shaky, fix it before tackling § 5 — that's where everything from Week 7 gets stress-tested.

### Notation used throughout

| Symbol | Meaning |
|---|---|
| $T$ | Sequence length |
| $d$ (or $D$) | Model / embedding dimension |
| $N$ | SSM latent state dimension |
| $x_t, y_t$ | Input / output at timestep $t$ |
| $h_t$ | Hidden state at timestep $t$ |
| $Q, K, V$ | Query, key, value matrices |
| $A, B, C$ | Continuous SSM matrices |
| $\bar A, \bar B$ | Discretised SSM matrices |


---

## § 1 · Quick Review (RNN · Self-Attention · Transformer)

This section is a condensed refresher. If you find yourself re-learning, pause and go back to Week 7 — the rest of this notebook assumes fluency with this material.

### § 1.1 The sequence-to-sequence task

Many real problems map a sequence of length $N$ to a sequence of length $M$:

$$x = (x_1, x_2, \ldots, x_N) \quad \longmapsto \quad y = (y_1, y_2, \ldots, y_M)$$

Canonical examples: machine translation (English words → French words), speech recognition (audio frames → characters), text summarisation (long document → short summary). In the general case $M \ne N$, and a given $y_t$ may depend on arbitrary positions of $x$.

The core modelling question of sequence architectures is: **given this mapping task, what inductive biases does the model use to relate positions in $x$ to positions in $y$?**


### § 1.2 Vanilla RNN

The classical answer — carry a single hidden-state vector $h_t$ through time:

$$h_t = \tanh(W_h\, h_{t-1} + W_x\, x_t), \qquad y_t = W_y\, h_t$$

```
x₁     x₂     x₃     x₄     …      x_T
│      │      │      │             │
▼      ▼      ▼      ▼             ▼
h₀ → [RNN] → [RNN] → [RNN] → [RNN] → … → [RNN]
           │      │      │              │
           ▼      ▼      ▼              ▼
           y₁     y₂     y₃             y_T
```

**Four key properties:**

1. **Infinite context window** — $h_t$ can in principle depend on arbitrarily old inputs.
2. **$O(T)$ sequential cost** — one step per token at training and inference.
3. **Not parallelisable along $T$** — each step needs $h_{t-1}$ from the previous step, so a GPU cannot spread the timesteps across cores.
4. **Constant state size** — $h_t$ is a fixed-size vector regardless of $T$.

**Two well-known failure modes** (Week 7 in detail):

- **Vanishing gradients**: backpropagating through time produces a product $\prod_{t} W_h \cdot \text{diag}\big(\tanh'(\cdot)\big)$. If the singular values of this product are $<1$, the gradient decays to zero exponentially — long-range dependencies can't be learned.
- **Exploding gradients**: the symmetric failure, with singular values $>1$.

LSTM/GRU tame the vanishing-gradient problem via gated additive state updates, but they remain fundamentally sequential along time.


### § 1.3 Self-attention is the fix for parallelisability

Replace "carry state through time" with "let every token look at every other token directly":

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

**What we gain:**

- **Parallelisable**: all timesteps are computed in one big matmul — no sequential dependency along $T$. Training goes from "one step per token" to "one matmul for the whole sequence". GPUs love this.
- **Direct long-range interaction**: every token can attend to every other token in **one** layer — the "effective receptive field" is the whole sequence from the first layer. No need to propagate information slowly through many RNN steps.

**What we pay:**

- **$O(T^2 \cdot d)$ cost**: the attention matrix $QK^\top$ is $T \times T$. Doubling the sequence length quadruples both compute and memory for attention. This is the bottleneck § 5 is about.
- **No built-in notion of position**: pure attention is permutation-equivariant. We have to add positional encodings (sinusoidal, learned, or rotary) to inject order information.

This trade-off — sequential-but-cheap (RNN) vs parallel-but-quadratic (attention) — is the central tension of Week 8. RWKV, Mamba, and other efficient architectures are all attempts to *have both*.


### § 1.4 The Transformer block

Week 7 built the Transformer from scratch. Recap of the block structure:

```
Input x  (B, T, D)
   │
   ├─────────┐                        <-- residual
   ▼         │
LayerNorm    │
   │         │
   ▼         │
Multi-Head Attention (Q, K, V = x, x, x)
   │         │
   ▼         │
   +─────────┘
   │
   ├─────────┐                        <-- residual
   ▼         │
LayerNorm    │
   │         │
   ▼         │
FFN (Linear → GELU → Linear)
   │         │
   ▼         │
   +─────────┘
   │
   ▼
Output   (B, T, D)
```

**Encoder block** — attention is unmasked (every token sees every other token).
**Decoder block** — attention is causally masked (token $t$ only sees $t' \le t$).

This will be the building block for BERT (encoder-only, § 2) and GPT (decoder-only, § 3).


### § 1.5 Why this week exists

Week 7 left us with the Transformer. Week 8 asks two follow-up questions:

1. **Can this architecture be specialised further?** — § 2 (BERT, encoder-only), § 3 (GPT, decoder-only), § 4 (ViT, Transformer applied to images). Each takes the Transformer block and commits to a different usage pattern.
2. **Can the $O(T^2)$ core attention operator be replaced?** — § 5 (SSM / Mamba, plus Appendix A's RWKV). Each proposes a different linear-cost sequence operator while trying to keep Transformer-level modelling power.

These are the two arrows of Week 8. Keep them in mind as you read — every architecture below is an answer to one or the other.


---

## § 2 · BERT (Bidirectional Encoder Representations from Transformers)

### § 2.1 From word embeddings to contextualised word embeddings (ELMo)

Early NLP models used **static word embeddings** — each word type has one fixed vector. Word2Vec (2013), GloVe (2014). The progression:

1. **1-of-N encoding** — one axis per word in a vocabulary of size $|V|$. Vectors are sparse, high-dimensional, and carry no similarity information: "dog" and "cat" are as different as "dog" and "tree".
2. **Static word embedding** — map each word to a dense vector (e.g. 300-D) learned such that semantically similar words are close. "dog" ≈ "cat", both far from "laptop". Word classes emerge automatically.
3. **Contextualised word embedding** — the same word gets a *different* vector depending on its context sentence.

Why step 3 is necessary — the classic **polysemy** example from the lecture slides. Consider the word **"bank"** across five sentences:

| Sentence | Meaning of "bank" |
| --- | --- |
| "Have you paid that money to the **bank** yet?" | financial institution |
| "It is safest to deposit your money in the **bank**." | financial institution |
| "The victim was found lying dead on the river **bank**." | riverside |
| "They stood on the river **bank** to fish." | riverside |
| "The hospital has its own blood **bank**." | storage facility |

One word, three distinct meanings. A static embedding must compress all three into a single vector — either averaging them (losing all distinctions) or arbitrarily picking one. What we want is a **different embedding each time "bank" appears**, conditioned on the surrounding words.

**ELMo** (Peters et al., NAACL 2018) provides this via a **bidirectional LSTM language model**:

```
              ← backward LSTM ←
              forward LSTM    →
x_1   x_2   x_3   …   x_T
 │     │     │         │
 ▼     ▼     ▼         ▼
h_1   h_2   h_3   …   h_T    ← each h_t depends on BOTH left and right context
```

Train a forward LSTM (predict $x_{t+1}$ from $x_{1:t}$) and a backward LSTM (predict $x_{t-1}$ from $x_{T:t}$) on raw text, then concatenate the two directions' hidden states at each position. The result is a context-dependent embedding for every token.

**What BERT does differently:** replace ELMo's bi-LSTM backbone with a **Transformer encoder stack**. This gives two immediate wins: (a) parallel training along $T$ (Transformer wins over LSTM), (b) fully bidirectional attention in a *single* pass instead of two separate directional passes that are only merged at the last layer.


### § 2.2 BERT architecture

BERT uses a stack of Transformer **encoder** blocks (identical to Week 7 § 6.5) **without** causal masking. Every token can attend to every other token — the attention is **bidirectional**.

**Special tokens:**

- `[CLS]`: prepended to every input. Its final hidden state serves as the "sentence representation" for classification tasks.
- `[SEP]`: separates two sentences in sentence-pair tasks.
- `[MASK]`: replaces tokens selected for prediction during pre-training.

**The key insight** — GPT's causal mask is necessary for **generation** (you can't see the future when generating), but it is **harmful** for **understanding** (you want to use both left and right context). BERT drops the causal mask.

**Input representation.** Each input token is the sum of three embeddings:

- **Token embedding** — looked up from the vocabulary (WordPiece, 30k types).
- **Segment embedding** — `A` or `B`, indicates which sentence the token belongs to (used in NSP).
- **Positional embedding** — learned, up to 512 positions.


### § 2.3 Masked Language Model (MLM)

**The problem with naive bidirectional LM.** If every token can see all other tokens *including itself*, it can trivially "cheat" by copying — the task becomes degenerate. You cannot train a standard left-to-right language model bidirectionally.

**BERT's solution — the Masked Language Model (MLM)**:

- Randomly select **15%** of tokens for prediction.
- Of those 15%:
  - **80%** are replaced with `[MASK]`
  - **10%** are replaced with a **random** token
  - **10%** are **kept unchanged**

**Why the 80/10/10 split?** This is a crucial and often-misunderstood design choice.

| Branch | What happens | Why |
| --- | --- | --- |
| 80% `[MASK]` | Token replaced with `[MASK]` | Main objective — predict the masked token from its context |
| 10% random | Token replaced with a random vocabulary word | Forces the model to maintain good representations for **every** token position, not just when it sees `[MASK]`. The model can't just ignore non-`[MASK]` positions during pre-training. |
| 10% unchanged | Token kept as-is | Reduces the **distribution mismatch** between pre-training (where `[MASK]` tokens appear often) and fine-tuning (where `[MASK]` never appears in real sentences) |

**Pre-training loss:**

$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in \mathcal{M}} \log P(x_i \mid \hat{x};\, \theta)$$

where $\mathcal{M}$ is the set of masked positions and $\hat x$ is the corrupted input sequence.


### § 2.4 Next Sentence Prediction (NSP)

BERT adds a **secondary objective**: given two sentences A and B, predict whether B is the **actual next sentence** following A (50% positive, 50% random negative). This is a binary classification on the `[CLS]` token's final hidden state.

**Input format for NSP:**

```
[CLS]  tok_a_1  tok_a_2  ...  [SEP]  tok_b_1  tok_b_2  ...  [SEP]
```

**Purpose:** teach BERT to model *relationships* between sentences, so that downstream tasks requiring sentence-pair reasoning (natural language inference, question answering, paraphrase detection) have a head-start.

**A historical note.** Later work (**RoBERTa**, Liu et al. 2019) showed that NSP may actually hurt performance — training on longer continuous spans with MLM alone often does better. This is a good illustration of how the field evolves: not every design choice in a seminal paper stands the test of time. BERT's NSP is still studied because it's historically important, but it is not a load-bearing ingredient.

**Approaches 1 and 2 (MLM + NSP) are used simultaneously** during pre-training — the same forward pass predicts masked tokens *and* the "next sentence" bit.


### § 2.5 How to use BERT — four fine-tune patterns

The same pre-trained BERT adapts to many downstream tasks by adding a simple **head** (a small task-specific module) and fine-tuning. The lecture slides 21–26 lay out **four** canonical patterns:

---

**Case 1 · Single sentence → class.** *Sentiment analysis, document classification, spam detection.*

```
[CLS]  w_1  w_2  w_3  ...  w_T
  │
  ▼
Linear classifier → class label
```

The `[CLS]` token's final hidden state is treated as the sentence representation; a linear layer on top produces the class logits. Fine-tune both BERT and the linear classifier end-to-end on the labelled dataset.

---

**Case 2 · Single sentence → per-token class.** *Slot filling, named entity recognition (NER), part-of-speech tagging.*

```
[CLS]  w_1  w_2  w_3  ...  w_T
       │    │    │         │
       ▼    ▼    ▼         ▼
       linear classifier (per token)
       │    │    │         │
       ▼    ▼    ▼         ▼
       y_1  y_2  y_3  ...  y_T
```

Apply a linear classifier at each token position independently. Every token gets its own label (e.g. `B-PER`, `I-PER`, `O` for NER). `[CLS]` is ignored.

---

**Case 3 · Sentence pair → class.** *Natural language inference (NLI), paraphrase detection, semantic similarity.*

```
[CLS]  premise_tokens  [SEP]  hypothesis_tokens  [SEP]
  │
  ▼
Linear classifier → {entailment, contradiction, neutral}
```

Feed both sentences into BERT in one pass, separated by `[SEP]`. The `[CLS]` token's final hidden state captures the joint representation. Linear classifier on top.

**Example task — NLI.** Given a *premise* and a *hypothesis*, decide whether the hypothesis is True, False, or Unknown with respect to the premise.

---

**Case 4 · Extraction-based question answering (SQuAD).** *Given a document and a question, mark the span inside the document that answers the question.*

```
[CLS]  question_tokens  [SEP]  document_tokens  [SEP]
                               │    │    │         │
                               ▼    ▼    ▼         ▼
                        (final hidden states h_i)
                               │    │    │         │
                               ▼    ▼    ▼         ▼
                            s₁, s₂, s₃, ... s_T    ← start logits = s · h_i
                            e₁, e₂, e₃, ... e_T    ← end logits   = e · h_i
                               │                   │
                               ▼                   ▼
                             softmax             softmax
                               │                   │
                               ▼                   ▼
                        answer span = [argmax start, argmax end]
```

Introduce two new learnable vectors: a **start vector** $s$ and an **end vector** $e$, both of dimension $D$. For each document token at position $i$, compute:

- start logit $= s \cdot h_i$
- end logit $\phantom{t} = e \cdot h_i$

Apply softmax over all document positions. The answer span is the $(i, j)$ with highest start$_i$ + end$_j$ subject to $i \le j$. During fine-tuning, supervise the gold start and end positions with cross-entropy loss.

This is what BERT does on SQuAD 1.1 — and it beats pre-BERT pipelines by a large margin.

---

**What the four cases share.** BERT itself is *not* task-specific. The same pre-trained weights feed four (and more) different heads, each a tiny linear layer or two. Fine-tune the whole stack, and 300M parameters of general-purpose language understanding become specialised to your task for free.


### § 2.6 Multilingual BERT (mBERT)

A single BERT model can be pre-trained on **many languages at once**. Google's *Multilingual BERT* (2018) is trained on Wikipedia text from **104 languages**, using a shared WordPiece vocabulary that covers scripts from Latin to Cyrillic to CJK.

**The surprising result:** mBERT exhibits strong **cross-lingual transfer**. Fine-tune it on an English NLI dataset, and it performs well on Chinese NLI at test time — *without ever seeing labelled Chinese data*. The shared representation space learnt during pre-training implicitly aligns semantically similar words across languages.

This was striking at the time because no explicit cross-lingual signal was used during training — the model just saw monolingual text from each language, mixed into one dataset. Pre-training on raw text alone is apparently enough to induce a loosely aligned multilingual representation space.

Follow-up work (XLM, XLM-R) added explicit cross-lingual objectives and scaled up training; these consistently beat mBERT at cross-lingual transfer. But mBERT is still a useful baseline and an illustrative example of the "pre-training is secretly doing a lot of work" phenomenon.


### § 2.7 Code demo — GPT (causal) vs BERT (bidirectional) attention masks

Visualise the difference between the causal attention mask used by GPT and the full attention mask used by BERT. Same sequence, two different masks.


In [ ]:
# ── GPT vs BERT attention patterns ───────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch

%matplotlib inline

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
T = 6
tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']

# GPT: causal (lower triangular)
gpt_mask = torch.tril(torch.ones(T, T))
ax1.imshow(gpt_mask, cmap='Blues', vmin=0, vmax=1)
ax1.set_title('GPT: Causal Attention', fontsize=13)
ax1.set_xticks(range(T)); ax1.set_xticklabels(tokens, fontsize=8)
ax1.set_yticks(range(T)); ax1.set_yticklabels(tokens, fontsize=8)
ax1.set_xlabel('Key'); ax1.set_ylabel('Query')

# BERT: bidirectional (full matrix)
bert_mask = torch.ones(T, T)
ax2.imshow(bert_mask, cmap='Oranges', vmin=0, vmax=1)
ax2.set_title('BERT: Bidirectional Attention', fontsize=13)
ax2.set_xticks(range(T)); ax2.set_xticklabels(tokens, fontsize=8)
ax2.set_yticks(range(T)); ax2.set_yticklabels(tokens, fontsize=8)
ax2.set_xlabel('Key'); ax2.set_ylabel('Query')

plt.tight_layout()
plt.show()


---

## § 3 · GPT (Generative Pre-trained Transformer)

### § 3.1 Architecture

GPT is the most natural extension of what we built in Week 7. Recall that Week 7 ended by training a **decoder-only Transformer** with **causal masking** for language modelling. GPT takes that exact recipe and asks: *what happens if we scale it up massively and pre-train it on enormous corpora?*

The architecture is simply a stack of $N$ Transformer decoder blocks, each consisting of:

- **Masked multi-head self-attention** — causal mask ensures each token only attends to previous tokens
- **Feed-forward network** — position-wise MLP
- **Layer normalisation** + **residual connections**

This is identical to Week 7's encoder block but with the causal mask **always applied**. GPT uses *only* the decoder side — there is no separate encoder. The lecture framing:

> "A Transformer uses an Encoder stack to model input and a Decoder stack to model output. If we only want to model the input (for downstream classification), we drop the decoder → **BERT**. If we have no input and only want to model the next word, we drop the encoder → **GPT**."

Both are specialisations of the same Transformer block, just with different masking and different training objectives.


### § 3.2 The pre-training + fine-tuning paradigm

**Pre-training objective** — standard causal language model:

$$\mathcal{L}_{\text{pretrain}} = -\sum_{t=1}^{T} \log P(x_t \mid x_1, \ldots, x_{t-1};\ \theta)$$

The model learns to predict the next token given all previous tokens. This is a **self-supervised** objective — no labelled data needed, just raw text.

**Fine-tuning.** After pre-training, add a task-specific head (e.g. a linear classifier) on top of the final hidden state and fine-tune on labelled data. This is directly analogous to ImageNet pre-training for CNNs (Week 5) — a single pre-trained model captures general knowledge that **transfers** to many downstream tasks.

| Phase | Data | Objective | Parameters |
| --- | --- | --- | --- |
| Pre-training | Large unlabelled corpus (Common Crawl, Wikipedia, books) | Next-token prediction | Train all |
| Fine-tuning | Small labelled dataset (task-specific) | Task-specific loss | Fine-tune all (or head only) |


### § 3.3 Scaling laws

A remarkable empirical finding (**Kaplan et al., 2020**): language model performance improves as a **power law** with model size, dataset size, and compute:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}$$

where $L$ is the test loss, $N$ is the number of parameters, $N_c$ is a critical parameter count, and $\alpha_N \approx 0.076$.

**What this says in plain language:** double the parameters, predictably reduce the loss by a fixed fraction. This is why GPT-2 (1.5B) → GPT-3 (175B) → GPT-4 (rumoured ~1T) showed consistent improvements with scale. The returns are diminishing (power law, not linear), but remarkably consistent and predictable.

**A note on scope.** Scaling laws are *not* in the Week 8 lecture slides explicitly — the lecture gestures at scale via the "ELMo 94M → BERT 340M → GPT 175B" size progression on slide 29. Scaling laws are included here because they are the quantitative story behind that progression and have become a standard part of the LLM literature.


In [ ]:
# ── Scaling law visualisation ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Simulated scaling law: L(N) = (Nc / N) ^ alpha
N_params = np.logspace(6, 11, 50)  # 1M to 100B parameters
Nc = 8.8e13   # critical parameter count (Kaplan et al. 2020)
alpha = 0.076
loss = (Nc / N_params) ** alpha

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_params, loss, 'b-', linewidth=2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Parameters (N)', fontsize=12)
ax.set_ylabel('Test Loss (L)', fontsize=12)
ax.set_title('Neural Scaling Law: Loss vs. Model Size', fontsize=13)

# Annotate GPT family
for name, n in [('GPT-2\n(1.5B)', 1.5e9), ('GPT-3\n(175B)', 1.75e11)]:
    l = (Nc / n) ** alpha
    ax.annotate(name, xy=(n, l), fontsize=9,
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(n * 3, l * 1.15), color='red')

ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()


### § 3.4 In-context learning (ICL)

Perhaps the most surprising discovery about large GPT models: they can **learn new tasks at inference time** from examples placed in the prompt, **without any gradient updates**.

- **Zero-shot**: `"Translate English to French: cheese =>"` — no examples given.
- **Few-shot**: provide 2–3 (input, output) examples in the prompt, then the query, all as a single text input.

The model performs implicit pattern recognition over the prompt — it identifies the task from the examples and applies it to the query. This is sometimes called "in-context learning" because *learning happens within the context window*, not through weight updates. Mechanistic interpretability research suggests that ICL works partly because Transformers can implement simple learning algorithms (like gradient descent or nearest-neighbour retrieval) *inside* a forward pass, using attention to read examples from the context.

**Why it matters.** ICL changed the economics of deploying language models: instead of fine-tuning a separate model for every task, you can use one large model and re-target it by rewriting the prompt. This is the property that made GPT-3 culturally important beyond academia.

**Scope note.** Like scaling laws, ICL is not explicitly on the Week 8 slides — it is included here because any serious treatment of GPT has to mention it.


In [ ]:
# ── Conceptual: In-context learning prompt construction ────────────

def build_icl_prompt(examples, query):
    """Construct a few-shot prompt for in-context learning.

    Args:
        examples: list of (input, output) pairs
        query:    the input to classify / translate / ...

    Returns:
        A single string prompt ready to feed into a language model.
    """
    prompt = ""
    for inp, out in examples:
        prompt += f"Input: {inp}\nOutput: {out}\n\n"
    prompt += f"Input: {query}\nOutput:"
    return prompt


# Example: sentiment classification via few-shot prompting
examples = [
    ("This movie was fantastic!", "Positive"),
    ("Terrible waste of time.", "Negative"),
    ("The acting was superb and the plot kept me engaged.", "Positive"),
]
query = "I fell asleep halfway through."

prompt = build_icl_prompt(examples, query)
print("=== Few-shot Prompt ===")
print(prompt)
print()
print("(A large LM would complete this with: Negative)")


### § 3.5 GPT vs BERT — summary

| Aspect | GPT | BERT |
| --- | --- | --- |
| Architecture | Decoder-only Transformer | Encoder-only Transformer |
| Attention | Causal (unidirectional) | Full (bidirectional) |
| Pre-training | Next-token prediction | MLM + NSP |
| Strength | **Generation** | **Understanding** |
| Fine-tuning | Task-specific head | Task-specific head |
| Can generate text? | Yes (autoregressive) | No (not designed for it) |
| Context usage | Left only | Full (both directions) |
| Key innovations | Scale + pre-train/fine-tune + in-context learning | Bidirectional pre-training via MLM |

> **Transition.** Both GPT and BERT rely on standard self-attention with $O(T^2 \cdot d)$ complexity. For a sequence of length $T = 100{,}000$ (a book), the attention matrix has $10^{10}$ entries per head per layer — clearly infeasible. We ended Week 7 noting this limitation. § 5 asks: can we achieve Transformer-quality modelling with **linear** complexity in $T$? Two fundamentally different approaches (Mamba and, in Appendix A, RWKV) answer yes, via different mathematics.


---

## § 4 · Vision Transformer (ViT)

The Transformer was designed for sequences of word tokens. Can it also handle **images**? Dosovitskiy et al. (ICLR 2021) gave a simple and surprising answer: yes, by turning the image into a sequence of small patches.

### § 4.1 Motivation

Until ViT, computer vision was dominated by **convolutional neural networks** (CNNs). CNNs bake in strong assumptions about images:

- **Locality** — a filter only looks at a small neighbourhood of pixels
- **Translation equivariance** — the same filter slides across the whole image
- **Hierarchical composition** — shallow layers learn textures, deep layers learn object parts

Transformers have none of these priors. They treat their input as a set of tokens with positional encodings, and learn spatial relationships from scratch via global self-attention. This seems like a handicap — why throw away all that useful image structure? The ViT paper's answer: **throw it away, use enough data, and the Transformer learns something better**.

### § 4.2 How to represent an image as a sequence

The lecture (slides 33–37) walks through four steps.

**Step 1 · Split the image into patches.** An $H \times W \times 3$ image becomes a grid of $P \times P$ non-overlapping patches, where $P$ is typically 16 (for a 224×224 image, that gives $(224/16)^2 = 196$ patches). Each patch is a little $P \times P \times 3$ tile of pixels.

**Step 2 · Flatten and linearly project (vectorise).** Each patch is flattened into a $3P^2$-dimensional vector (e.g. $3 \cdot 16 \cdot 16 = 768$) and projected through a learnable linear layer to the model dimension $D$ (typically 768 for ViT-Base). The output is a single $D$-vector per patch — a **patch token**.

**Step 3 · Add positional encoding.** Unlike language, the "natural order" of patches is 2-D (not 1-D), but ViT just flattens the grid in row-major order and adds a **learned 1-D positional embedding** at each position. This is surprisingly enough — the model figures out the 2-D structure from data.

**Step 4 · Prepend a `[CLS]` token.** Following BERT, a learnable `[CLS]` token is prepended to the sequence. Its final hidden state feeds into a classification head.

The output sequence is then fed into a **standard Transformer encoder stack** (the same Week 7 block, no modifications). A linear classifier on the `[CLS]` token's final hidden state produces class logits.


### § 4.3 Code demo — patch embedding from scratch

The patchify-and-project step can be implemented as a **single 2-D convolution** with kernel size equal to stride equal to $P$. The conv's output at each spatial location is *exactly* the linear projection of the corresponding patch — so we get patchify + linear-project + no-overlap all in one op. This is the standard `PatchEmbed` module used across ViT implementations.


In [ ]:
# ── ViT patch embedding ─────────────────────────────────────────────────
import torch
import torch.nn as nn


class PatchEmbed(nn.Module):
    """ViT-style patch embedding.

    Splits an image into P×P non-overlapping patches, flattens each,
    and linearly projects to dim D.

    Implementation trick: a single Conv2d with kernel_size = stride = P.
    The conv's output at each spatial location IS the linear projection
    of the corresponding patch — patchify + linear-project in one op.
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)                           # (B, D, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)           # (B, T, D) with T = (H/P)*(W/P)
        return x


# Smoke test — standard ViT-Base/16 dimensions
embed = PatchEmbed(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
img = torch.randn(2, 3, 224, 224)
tokens = embed(img)

print(f"Input image:  {tuple(img.shape)}")
print(f"Patch tokens: {tuple(tokens.shape)}  (batch, T, D)")
print(f"Sequence length T = (224/16)^2 = {(224 // 16) ** 2}")
print(f"Parameters in PatchEmbed: {sum(p.numel() for p in embed.parameters()):,}")


### § 4.4 The ViT architecture in one diagram

```
Input image (224, 224, 3)
        │
        ▼
  PatchEmbed              (Conv2d with kernel=stride=16)
        │
        ▼
  Patch tokens: 196 × 768
        │
        ▼
  Prepend [CLS]           → 197 × 768
        │
        ▼
  + Positional embedding  (learned, 197 × 768)
        │
        ▼
  Transformer Encoder × L (standard Week 7 block, unmasked attention)
        │
        ▼
  Take [CLS] token's final hidden state → 768
        │
        ▼
  Linear classifier head  → class logits (e.g. 1000 classes for ImageNet)
```

That's it — ViT is genuinely just "tokenise the image and feed it into a Transformer". The whole architectural novelty fits in `PatchEmbed`. Everything after that is pure Week 7 material.


### § 4.5 Data hunger — the inductive bias ceiling

**The catch.** On **ImageNet-1K** alone (~1.2M training images, 1000 classes), ViT **underperforms** a comparably-sized CNN. The slide 40 headline from the lecture: EfficientNet-B7 (a strong CNN) gets 84.3% top-1 on ImageNet-1K, and vanilla ViT-Large gets *less* when trained only on ImageNet-1K.

But when pre-trained on **JFT-300M** (~300M images, Google's internal dataset) and then fine-tuned on ImageNet-1K, ViT *surpasses* the best CNN by a meaningful margin. More data unlocks ViT; with less data, ViT trails.

**Why this pattern?** Inductive bias. CNNs' built-in assumptions (locality, translation equivariance, hierarchical composition) are *exactly right* for natural images — the priors match the data. ViT has none of these priors, so it has to **learn them from data**.

- On **small datasets**, CNN's priors do the work for free; ViT cannot see enough examples to rediscover the same structure.
- On **huge datasets**, ViT has enough examples to learn richer, more flexible spatial organisations than the hard-coded CNN priors allow — including global, long-range dependencies in a single layer — and the rigid CNN structure becomes a *ceiling* that ViT can now exceed.

**In one line:** inductive bias is a double-edged sword. It does your homework for you when data is scarce, and holds you back when data is abundant. This is also Exam Q3 (b) in the Week 8 tutorial.

**Numerical exercise — the patch-size trade-off.** Suppose you change the patch size from $P=16$ to $P=32$. Work out what happens to:

1. The sequence length $T = (224 / P)^2$
2. The single-layer self-attention FLOPs, which scale as $T^2 \cdot d$
3. The spatial area each token covers ($P^2$ pixels)

Answer: $T$ drops from 196 to 49 (4× shorter), FLOPs drop by $(196/49)^2 = 16\times$, and each token covers $1024$ pixels instead of $256$ (4× coarser). Larger patches → cheaper attention + coarser spatial granularity. Smaller patches → more expressive + quadratically more expensive.


### § 4.6 Swin Transformer — adding locality back in

Vanilla ViT has *global* attention at every layer — every patch attends to every patch — so a single layer is $O(T^2)$ where $T$ is the number of patches. For a 224×224 image at patch size 16 that's $T=196$, which is fine; but dense-prediction tasks like semantic segmentation or object detection want much higher spatial resolution (think 1024×1024 or more). At that resolution, $T$ explodes and global attention becomes infeasible.

**Swin Transformer** (Liu et al., ICCV 2021 *Best Paper*) puts locality back in. Two ideas:

1. **Window attention.** Restrict self-attention to local, non-overlapping $M \times M$ windows of patches (e.g. $M=7$). Each window does its own self-attention in parallel. Complexity becomes $O(M^2 \cdot T)$, which is **linear in the number of patches** — you can scale to arbitrarily large images.

2. **Shifted windows.** Windowed attention alone has no way to mix information *across* windows. Swin fixes this by **shifting the window grid by half a window** at alternating layers. A patch that lived in window $A$ at layer $\ell$ lives in a *different* window $A'$ at layer $\ell+1$, so information flows across window boundaries as depth grows. This is the "shifted windows" in the paper title.

**Plus** a hierarchical feature pyramid: Swin merges adjacent patches into one at every stage (like the downsampling in a CNN), building a multi-scale representation that can be plugged directly into detection and segmentation heads designed for CNNs.

**Net effect:** Swin gives you Transformer-level modelling flexibility with CNN-style spatial efficiency and CNN-style hierarchical features — enabling state-of-the-art on ImageNet classification *and* COCO detection *and* ADE20K segmentation in one architecture family. It won the ICCV 2021 best paper.

**Intellectually**, Swin is the dual move to Mamba: Mamba keeps the Transformer's structure and replaces its *attention operator* with something cheaper; Swin keeps the *attention operator* and restricts its *scope* (from global to windowed) to match the locality structure of natural images. Both aim at sub-quadratic cost; they get there from different directions.


---

## § 5 · Efficient sequence modelling — State Space Models and Mamba

This is the most technically dense section of the notebook. It is also the subject of the live tutorial — if you came from the tutorial with the recurrent ↔ convolutional duality feeling shaky, this section is where you consolidate it.

### § 5.1 The $O(T^2)$ pain, revisited

Standard self-attention computes a $T \times T$ attention matrix. As sequences grow longer this becomes the dominant bottleneck. Concrete numbers from the lecture slide 43:

- GPT-3 context window: **4K tokens**
- GPT-4 context window: **32K tokens**
- GPT-5 (rumoured): **~200K tokens**

Doubling $T$ quadruples the attention matrix — memory and compute both grow as $T^2$. At $T = 128{,}000$ with $d = 4096$ (a typical large-LM configuration), a single attention layer's matrix occupies $128{,}000^2 \cdot 2\text{ bytes} \approx 33$ GB for fp16 — larger than most GPUs' total memory. And this is per layer, per head.

The question this section answers is: **can we build a sequence operator that is $O(T)$ (or $O(T \log T)$) instead of $O(T^2)$, without losing Transformer-level expressiveness?**


In [ ]:
# ── O(T²) vs O(T) scaling ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

T_vals = np.arange(100, 100001, 100)
d = 512

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_vals, T_vals**2 * d,
        label=r'$O(T^2 \cdot d)$ — Standard Attention', color='#e74c3c')
ax.plot(T_vals, T_vals * d**2,
        label=r'$O(T \cdot d^2)$ — SSM / RWKV / linear-cost sequence ops',
        color='#2ecc71')
ax.set_xlabel('Sequence Length T', fontsize=12)
ax.set_ylabel('Operations', fontsize=12)
ax.set_yscale('log')
ax.set_title('Why $O(T^2)$ Attention Cannot Scale to Long Sequences',
             fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axvline(x=d, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'T = d = {d}\n(break-even)', xy=(d, d**3), fontsize=9, color='gray')
plt.tight_layout()
plt.show()


### § 5.2 State Space Models from first principles

**Origin.** State Space Models (SSMs) come from classical **control theory** — Kalman's 1960 paper *"A New Approach to Linear Filtering and Prediction Problems"* introduced the formalism that is still used today in signal processing, aerospace guidance, and econometrics.

A continuous-time linear SSM models a hidden state $h(t) \in \mathbb{R}^N$ evolving under an input $x(t)$, observed through an output $y(t)$:

$$\boxed{\,h'(t) = A\, h(t) + B\, x(t), \qquad y(t) = C\, h(t) + D\, x(t)\,}$$

where:

- $h(t) \in \mathbb{R}^N$ is the hidden state (the "memory"), $N$ = state dimension (typically 16–64 in SSM literature)
- $x(t)$ is the input, $y(t)$ is the output
- $A \in \mathbb{R}^{N \times N}$ is the **state transition** matrix
- $B \in \mathbb{R}^{N \times 1}$ injects input into the state
- $C \in \mathbb{R}^{1 \times N}$ reads output from the state
- $D \in \mathbb{R}$ is an optional direct feedthrough term (often dropped for neural SSMs; we will drop it from here on)

The state equation says: *the state changes at a rate that is linear in itself plus linear in the input*. The output equation says: *what we observe is a linear readout of the state*.

**Crucially, this is a *linear* dynamical system**. Linear means the solution has a **closed form** — we can compute $y(t)$ without stepping through time one tiny step at a time. This is the property that § 5.4 will exploit to turn a recurrent model into a convolution.

### § 5.3 Discretisation — from continuous to discrete time

To apply an SSM to discrete token sequences, we need to discretise. Two standard choices:

**Option 1 — Euler (first-order).** Approximate $h'(t) \approx (h_t - h_{t-1}) / \Delta$:

$$h_t = h_{t-1} + \Delta (A h_{t-1} + B x_t) = (I + \Delta A) h_{t-1} + \Delta B x_t$$

Set $\bar A = I + \Delta A$ and $\bar B = \Delta B$. This is the choice the lecture uses (slides 45–46); simple, but low accuracy for stiff systems.

**Option 2 — Zero-Order Hold (ZOH).** Assume the input $x(t)$ is constant between samples (a "staircase" signal) and integrate the ODE exactly over one step. This gives

$$\bar A = e^{\Delta A}, \qquad \bar B = (\Delta A)^{-1}(e^{\Delta A} - I)\, \Delta B$$

More accurate, and this is the discretisation used in **S4** and **Mamba**. For small $\Delta$, $e^{\Delta A} \approx I + \Delta A$, so the two schemes agree to first order — the lecture's Euler pedagogy and the literature's ZOH yield the same discrete SSM equation:

$$\boxed{\,h_t = \bar A\, h_{t-1} + \bar B\, x_t, \qquad y_t = C\, h_t\,}$$

> **Key insight**: whatever discretisation you pick, you end up with a **linear RNN**. The matrix $\bar A$ plays the role of the hidden-to-hidden weight of a classical RNN. The specific structure of $\bar A$ inherited from the continuous $A$ is what makes SSMs interesting (§ 5.5–5.6), but structurally we are just back at an RNN — without the non-linearity.


### § 5.4 The two views — recurrent ↔ convolutional

This is the heart of the section. The same discrete SSM admits two computationally different but mathematically equivalent evaluation strategies.

**View 1 — Recurrent.** Evaluate token by token, carrying the state:

$$h_t = \bar A\, h_{t-1} + \bar B\, x_t, \qquad y_t = C\, h_t$$

Properties: **$O(d)$ memory per step** (only $h_t$ needs to be held), **$O(T)$ total time**, **not parallelisable** along $T$ (each step waits for the previous one). This is exactly like an RNN at inference time.

**View 2 — Convolutional.** Assume $h_0 = 0$ and unroll the recurrence:

$$h_1 = \bar B\, x_1$$
$$h_2 = \bar A\, h_1 + \bar B\, x_2 = \bar A \bar B\, x_1 + \bar B\, x_2$$
$$h_3 = \bar A\, h_2 + \bar B\, x_3 = \bar A^2 \bar B\, x_1 + \bar A \bar B\, x_2 + \bar B\, x_3$$
$$\vdots$$
$$h_t = \sum_{i=0}^{t} \bar A^{\,t-i}\, \bar B\, x_i$$

Apply the output projection:

$$y_t = C\, h_t = \sum_{i=0}^{t} \big(C\, \bar A^{\,t-i}\, \bar B\big)\, x_i$$

Define the kernel $K$ of length $T$ by

$$K_i = C\, \bar A^{\,i}\, \bar B \in \mathbb{R}$$

Then

$$y_t = \sum_{i=0}^{t} K_{t-i}\, x_i = (K * x)_t$$

— **a single 1-D causal convolution**.

**Properties:** the full kernel $K$ can be computed once at training time (cost $O(T \cdot d^2)$ naively for matrix powers, reducible to $O(\log T)$ with repeated squaring and $O(T)$ with clever structure in $\bar A$). The convolution itself is computed in $O(T \log T)$ via FFT, or $O(T^2)$ naively with a double loop, and is **fully parallel along $T$** — exactly what GPUs want for training.

**Three equivalent views, one SSM:**

| View | Formula | Per-token cost | Parallelisable? | Use for |
| --- | --- | --- | --- | --- |
| **Recurrent** | $h_t = \bar A h_{t-1} + \bar B x_t$ | $O(d^2)$, $O(d)$ state | No | Inference (streaming) |
| **Convolution** | $y = K * x$, $K_i = C \bar A^i \bar B$ | $O(d^2)$ amortised | Yes (FFT, $O(T \log T)$) | Training |
| **Linear attention** (see also) | — | $O(T \cdot d)$ | Yes (assoc. scan) | Alternative training path |

**The key idea.** *The same operator is an RNN at inference time and a convolution at training time.* You train with the convolutional view to get GPU parallelism, then switch to the recurrent view for O(1)-memory streaming inference. No model change — just two ways to compute the same thing.

**Code demo.** Below is a worked numerical demo that verifies both forms give identical outputs on a random SSM. This is the same calculation as the live tutorial's Task A / Task B — run it, print the outputs, and convince yourself the duality is real.


In [ ]:
# ── SSM two-view duality demo ──────────────────────────────────────────────
import torch
import torch.nn.functional as F

torch.manual_seed(1)
T, d = 30, 4

# Random stable SSM (spectral radius of A_bar < 1)
A_bar = 0.9 * torch.eye(d) + 0.02 * torch.randn(d, d)
B_bar = torch.randn(d, 1)
C_bar = torch.randn(1, d)
x = torch.randn(T)


def ssm_recurrent(A_bar, B_bar, C_bar, x):
    """View 1: recurrent form (RNN-style, O(1) state)."""
    T, d = x.shape[0], A_bar.shape[0]
    h = torch.zeros(d)
    y = torch.zeros(T)
    b = B_bar.squeeze(-1)
    for t in range(T):
        h = A_bar @ h + b * x[t]
        y[t] = (C_bar @ h).squeeze()
    return y


def ssm_convolutional(A_bar, B_bar, C_bar, x):
    """View 2: convolutional form (y = K * x)."""
    T = x.shape[0]

    # Build K[i] = C @ A^i @ B incrementally — maintain running A^i @ B.
    K = torch.zeros(T)
    A_pow_B = B_bar.clone()              # starts at A^0 @ B = B
    for i in range(T):
        K[i] = (C_bar @ A_pow_B).squeeze()
        A_pow_B = A_bar @ A_pow_B

    # Causal convolution via torch.nn.functional.conv1d.
    # conv1d cross-correlates, so flip the kernel to get true convolution.
    x_pad = F.pad(x.view(1, 1, T), (T - 1, 0))      # (1, 1, 2T-1)
    K_flip = K.flip(0).view(1, 1, T)
    y = F.conv1d(x_pad, K_flip).view(T)
    return y


y_rec  = ssm_recurrent(A_bar, B_bar, C_bar, x)
y_conv = ssm_convolutional(A_bar, B_bar, C_bar, x)

max_diff = (y_rec - y_conv).abs().max().item()
print(f"recurrent form :  y[:5] = {y_rec[:5].tolist()}")
print(f"conv form      :  y[:5] = {y_conv[:5].tolist()}")
print(f"max abs diff   :  {max_diff:.2e}")
assert torch.allclose(y_rec, y_conv, atol=1e-4), "forms disagree"
print()
print("✓ recurrent and convolutional forms agree on T=30")


### § 5.5 HiPPO — initialising $A$ so the model remembers

The closed-form unroll $h_t = \sum_{i\le t} \bar A^{\,t-i}\bar B x_i$ makes the role of $\bar A$ painfully concrete: the contribution of input $x_i$ to state $h_t$ is weighted by $\bar A^{\,t-i}$. If the spectral radius of $\bar A$ is $<1$, those weights **decay exponentially**: inputs from $t-100$ steps ago are essentially invisible. If the spectral radius is $>1$, they **explode**. Random initialisation of $\bar A$ gives you one or the other — neither is what we want.

**HiPPO** (*High-order Polynomial Projection Operator*, Gu et al. NeurIPS 2020) solves this constructively. The idea: pick $A$ so that the state $h_t$ **optimally approximates the entire input history** $x_{1:t}$ *as a projection onto an orthogonal polynomial basis*.

More concretely — fix a basis of orthogonal polynomials $\{p_k\}_{k=0}^{N-1}$ on the interval $[0, t]$ (e.g. **Legendre polynomials** for HiPPO-LegS). Define $h_t$ to be the vector of coefficients when we project the input history $x_{[0,t]}$ onto this basis:

$$h_t^{(k)} = \int_0^t p_k(\tau) x(\tau)\, d\tau$$

Differentiate this with respect to $t$ and you get an ODE of the form $h'(t) = A h(t) + B x(t)$ — an SSM! And the matrix $A$ that makes this work for the Legendre basis has a specific closed form:

$$A_{nk}^{(\text{LegS})} = \begin{cases} -(2n+1)^{1/2}(2k+1)^{1/2}, & n > k \\ -(n+1), & n = k \\ 0, & n < k \end{cases}$$

(This is a lower-triangular matrix with a specific, derivable structure — you don't need to memorise it, just know that the structure is inherited from the orthogonality of Legendre polynomials.)

**Why this helps.** The approximation error of a degree-$N$ polynomial projection onto a growing interval is bounded by standard results in numerical analysis — meaning the state $h_t$ provably captures the essential "shape" of the input history up to time $t$, with error that decays as $N$ grows. In plain terms: **HiPPO gives you long memory by construction.** It is why S4 can remember things that happened 10,000 tokens ago while a randomly-initialised linear RNN forgets everything after ~50 steps.

One-line takeaway: *HiPPO is not just a smart initialisation — it is the mathematical reason why SSMs work at all for long sequences.*

### § 5.6 S4 — Structured State Space for Sequence modelling

**S4** (Gu, Goel, Ré, ICLR 2022, *Outstanding Paper*) combines three ingredients into the first SSM architecture that beat Transformers on the Long Range Arena benchmark:

1. **HiPPO initialisation** (§ 5.5) for long memory.
2. **Structured $A$** — specifically, $A$ is parameterised as **diagonal plus low-rank** (DPLR). This lets the matrix power $\bar A^i$ be computed efficiently via **fast kernel algorithms** based on Cauchy matrices and a clever reparameterisation in the frequency domain. Without this, computing the $T$-length kernel $K$ would dominate training cost.
3. **FFT-based convolution** — once $K$ is computed, the convolution $y = K * x$ is evaluated in $O(T \log T)$ via FFT.

**Result.** S4 was the first SSM to reach Transformer-level quality on language-like benchmarks, plus it handled very long sequences (path finding, Markov chain predictions at $T = 16{,}384$) where standard Transformers struggled due to quadratic cost. It is the direct mathematical ancestor of Mamba.

**Limitation.** Like every SSM before Mamba, S4's $\bar A, \bar B, C$ matrices are **input-independent constants**. Every token is filtered through the same linear system. This limits expressiveness — you can't "skip" an uninformative token, or dynamically reset memory based on content. Fixing this is Mamba's contribution.


### § 5.7 Mamba (S6) — adding selection

#### § 5.7.1 The motivation for selection

Standard SSMs (S4 and its relatives) use **fixed** $A$, $B$, $C$, $\Delta$ matrices. They process every token identically regardless of content. This is pleasant mathematically (it preserves the convolution form) but limits expressiveness — the model cannot decide *on a per-token basis* how much of the current input to absorb or how much of the existing state to keep.

**Mamba's insight** (Gu & Dao, COLM 2024; ~10,000 citations): make $B$, $C$, and the step size $\Delta$ **input-dependent**. Instead of learned constants, they become *linear functions of $x_t$*:

$$B_t = \text{Linear}_B(x_t), \qquad C_t = \text{Linear}_C(x_t), \qquad \Delta_t = \text{softplus}\!\big(\text{Linear}_\Delta(x_t)\big)$$

Now the model can **selectively** decide per token:

- **What to absorb** — $B_t$ controls how much of $x_t$ enters the state
- **What to read out** — $C_t$ controls what to emit from the state
- **How fast time passes** — $\Delta_t$ controls the effective step size of the discretisation

**Connection to LSTM gating.** The role of $\Delta_t$ is strikingly similar to an LSTM forget gate:

| $\Delta_t$ | $\bar A_t = e^{\Delta_t A}$ | Effect | LSTM analogue |
| --- | --- | --- | --- |
| $\to \infty$ | $\to 0$ | State **resets** (forget everything) | forget gate $\approx 0$ |
| $\to 0$ | $\to I$ | State **preserved** (remember everything) | forget gate $\approx 1$ |

Mamba learns this per-token control over memory — just like LSTM, but within the SSM framework, and trained in parallel.

#### § 5.7.2 What this breaks — the convolution form disappears

The convolution form required $\bar A, \bar B, C$ to be **constants**: only then does $K_i = C \bar A^i \bar B$ depend on the gap $t - i$ alone, which is the definition of a shift-invariant kernel. If every timestep has its own $\bar A_t, \bar B_t, C_t$, then the "kernel" would be different at every output position $t$ — not a shift-invariant operator at all, so it cannot be written as a single convolution.

This is a hard trade-off: **Mamba gains input-dependence, but loses the FFT-based convolutional training path.** So how does it recover parallel training on GPUs?

#### § 5.7.3 Parallel scan recovers parallelism

**Key observation.** Even though $\bar A_t$ depends on $t$, the update rule $h_t = \bar A_t h_{t-1} + \bar B_t x_t$ is still a **linear** function of the previous state. Each per-step operator $(h_{t-1}) \mapsto (\bar A_t h_{t-1} + \bar B_t x_t)$ is an affine map, and **composition of affine maps is associative**. That means we can use a **parallel prefix scan** (Blelloch 1990, a classic algorithm from GPU programming) to compute $h_{1:T}$ in $O(\log T)$ depth with $O(T)$ work.

Concretely: pair up adjacent timesteps, compose their affine operators, recurse. After $\log_2 T$ levels of pairwise composition, you have the operator mapping $h_0$ to each $h_t$. This is a textbook associative-scan pattern, implemented on GPU with a few hundred lines of CUDA and matching the depth of the FFT-based S4 training path.

**Summary of the trade-offs:**

| Architecture | Parameters | Training | Inference |
| --- | --- | --- | --- |
| S4 | Input-independent $A, B, C$ | Convolution via FFT, $O(T \log T)$ depth | Recurrent, $O(1)$ state |
| Mamba (S6) | Input-dependent $B_t, C_t, \Delta_t$ | Parallel scan, $O(\log T)$ depth | Recurrent, $O(1)$ state |

Mamba trades the clean kernel $K$ for a log-depth tree of pairwise compositions — and gains per-token gating in exchange.

#### § 5.7.4 Hardware-aware state expansion

Naively implementing Mamba's scan is slow on GPU because the latent state tensor $h \in \mathbb{R}^{d_{\text{model}} \times N}$ (model-dim × state-dim) is much larger than the model-dim-only state of a standard Transformer — materialising it at every timestep costs global-memory bandwidth.

Mamba's trick: **never materialise the full state in global memory**. The scan is implemented as a fused CUDA kernel that keeps the state in **SRAM** (on-chip fast memory) throughout the scan, only writing the output $y_t$ back to global memory. This means the *state dimension* $N$ can be much larger (e.g. 16–64) than the model can afford if $h$ had to live in global memory — and larger $N$ means more memory capacity per token.

This is what the paper calls "hardware-aware state expansion". It's why Mamba's paper emphasises IO cost and SRAM utilisation — the mathematical algorithm only wins in practice when the kernel is written to match GPU memory hierarchy.

#### § 5.7.5 Code — a simplified Mamba block

Below is a **simplified** educational implementation. It uses an explicit Python loop for the scan (so it's slow) and doesn't do hardware-aware memory optimisations (so it's memory-inefficient). The goal is clarity, not speed — you should be able to read it end-to-end and see each concept from § 5.7.1–§ 5.7.4.


In [ ]:
# ── MAMBA: Selective SSM and block (simplified, educational) ──────────────
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelectiveSSM(nn.Module):
    """MAMBA selective state space model (simplified)."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        # A initialised with log-spaced values for stability
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A).unsqueeze(0).expand(d_model, -1).clone())
        self.D = nn.Parameter(torch.ones(d_model))  # skip connection
        # Input-dependent projections (the SELECTION mechanism)
        self.proj_B = nn.Linear(d_model, d_state, bias=False)
        self.proj_C = nn.Linear(d_model, d_state, bias=False)
        self.proj_delta = nn.Linear(d_model, d_model, bias=True)

    def forward(self, x):
        """x: (B, T, D) -> (B, T, D)"""
        B_batch, T, D = x.shape
        N = self.d_state
        A = -torch.exp(self.A_log)                   # (D, N) -- negative for stability

        # Input-dependent parameters
        B_t = self.proj_B(x)                         # (B, T, N)
        C_t = self.proj_C(x)                         # (B, T, N)
        delta = F.softplus(self.proj_delta(x))       # (B, T, D) step sizes

        # Discretise: delta_A = exp(delta * A)
        delta_A = torch.exp(                          # (B, T, D, N)
            delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0)
        )
        delta_B = delta.unsqueeze(-1) * B_t.unsqueeze(2)   # (B, T, D, N)

        # Sequential scan (recurrent mode — Python loop for clarity)
        h = torch.zeros(B_batch, D, N, device=x.device)    # (B, D, N)
        outputs = []
        for t in range(T):
            h = delta_A[:, t] * h + delta_B[:, t] * x[:, t].unsqueeze(-1)
            y_t = (h * C_t[:, t].unsqueeze(1)).sum(-1)     # (B, D)
            outputs.append(y_t)

        y = torch.stack(outputs, dim=1)              # (B, T, D)
        return y + x * self.D                        # skip connection


class MambaBlock(nn.Module):
    """Simplified MAMBA block."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner = d_model * expand
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)  # two paths
        self.conv1d = nn.Conv1d(d_inner, d_inner, d_conv,
                                 padding=d_conv - 1, groups=d_inner)
        self.ssm = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x):
        """x: (B, T, D) -> (B, T, D)"""
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)                         # (B, T, 2*d_inner)
        x, z = xz.chunk(2, dim=-1)                   # split into two paths
        # Path 1: Conv -> SiLU -> SSM
        x = self.conv1d(x.transpose(1, 2))[:, :, :x.shape[1]].transpose(1, 2)
        x = F.silu(x)
        x = self.ssm(x)
        # Path 2: Gating
        x = x * F.silu(z)
        return residual + self.out_proj(x)


In [ ]:
# ── MAMBA verification ──────────────────────────────────────────────────────
torch.manual_seed(42)
d_model = 64
mamba_block = MambaBlock(d_model, d_state=16, d_conv=4, expand=2)
x = torch.randn(2, 20, d_model)  # batch=2, seq_len=20
out = mamba_block(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}')
print(f'Parameters:   {sum(p.numel() for p in mamba_block.parameters()):,}')


#### § 5.7.6 Two alternative linear-attention routes — Mamba vs RWKV

Mamba is not the only way to get linear-cost sequence modelling. A second major family, **RWKV**, takes a different mathematical starting point (linearised exponential attention with time decay) and arrives at a similar parallel-train/recurrent-infer property. Appendix A covers it in detail. Side-by-side comparison:

| Aspect | RWKV (Appendix A) | Mamba / S6 (this section) |
| --- | --- | --- |
| **Mathematical basis** | Linearised softmax attention | State space models from control theory |
| **Ancestor** | Transformer + RNN | Control theory (Kalman 1960) → S4 |
| **Training mode** | Parallel (linear attention form) | Parallel (associative scan, $O(\log T)$ depth) |
| **Inference mode** | Recurrent ($O(1)$ memory per step) | Recurrent ($O(1)$ memory per step) |
| **Input dependence** | Partial — token-shift interpolation, fixed per-channel decay $w$ | Full — $B_t, C_t, \Delta_t$ all functions of $x_t$ |
| **Time complexity** | $O(T \cdot d)$ | $O(T \cdot d \cdot N)$, $N$ = state dim |
| **Key parameters** | $w$ (decay), $u$ (current-token bonus) | $A$ (dynamics), $\Delta_t$ (per-token step size) |
| **Gating mechanism** | Receptance gate $r_t$ | $\Delta_t$-controlled forget/update |
| **Positional encoding** | Implicit via time-decay $w$ | Implicit via discretisation |

Both are "linear-cost sequence operators that train in parallel and stream in constant memory". They differ in whether the mathematical object they generalise is *softmax attention* (RWKV) or *an ODE* (Mamba). Either route works; the two are currently actively competing in the research literature.

### § 5.8 Mamba 2, Vision Mamba, and MambaOut

**Mamba 2** (Dao & Gu, 2024). A reformulation of selective SSM as a **structured matrix product** with a specific mask pattern. The key observation: the selective scan can be re-expressed so that the computational primitive is the exact same matmul kernel that Transformers use — letting Mamba 2 reuse decades of highly-optimised matmul infrastructure (including tensor cores, FlashAttention-style IO optimisations). Net effect: similar modelling capacity to Mamba, much better hardware utilisation. The connection between SSMs and structured attention is called "state space duality".

**Vision Mamba** (ICML 2024). Applies selective SSM to image patches as an alternative to ViT. Two main challenges:

1. Images are 2-D, not causally ordered — Vision Mamba uses a **bidirectional scan** (one forward, one backward) to see both spatial directions, instead of a single causal scan.
2. The parallel scan kernel needs to be adapted for 2-D spatial traversal orders.

**LocalMamba** (slide 59 of the lecture). A follow-up that adds local windows on top of Vision Mamba — mirroring the ViT → Swin evolution — for better locality and hardware efficiency.

**MambaOut** (CVPR 2025, slide 61). A provocative follow-up paper: the authors show that a *simple gated CNN* with comparable compute budget matches Vision Mamba on ImageNet classification. The title is a pun on "Mamba" + "out (of scope)" — the authors argue Mamba's inductive bias does not help much for image classification specifically, though it remains useful for long-sequence language modelling. The broader lesson: **architectural wins do not always transfer across modalities**. An operator that is great for autoregressive text may be unnecessary when the task has strong spatial structure that a CNN already captures.


---

## § 6 · Grand comparison

### § 6.1 Unified comparison table

| Feature | GPT | BERT | ViT | RWKV (App. A) | Mamba | KAN (App. B) |
| --- | --- | --- | --- | --- | --- | --- |
| **Architecture type** | Decoder-only Transformer | Encoder-only Transformer | Encoder-only Transformer on image patches | Linear attention hybrid | Selective SSM | Alternative MLP block |
| **Core innovation** | Pre-train + scale + ICL | Masked LM (bidirectional) | Patchify → standard Transformer | WKV operator (dual-mode) | Input-dependent SSM | Learnable edge activations |
| **Attention mechanism** | Causal self-attention | Full self-attention | Full self-attention (windowed in Swin) | Linear attention (no softmax) | None (state space) | N/A |
| **Per-layer complexity** | $O(T^2 \cdot d)$ | $O(T^2 \cdot d)$ | $O(T^2 \cdot d)$ | $O(T \cdot d)$ | $O(T \cdot d \cdot N)$ | N/A |
| **Training mode** | Parallel | Parallel | Parallel | Parallel (linear attention) | Parallel (assoc. scan) | Standard backprop |
| **Inference mode** | Autoregressive, $O(T^2)$ via KV-cache | Bidirectional (one forward pass) | One forward pass | Recurrent, $O(1)$/step | Recurrent, $O(1)$/step | Standard forward pass |
| **Long-range deps** | Excellent (within context) | Excellent (full context) | Excellent (global attn) | Good (exp. decay) | Excellent (HiPPO-inspired) | N/A |
| **Interpretability** | Low | Low | Low | Low | Low | **High** (plot edge functions) |
| **Maturity** | Very high (GPT-4/5) | Very high (widely deployed) | Very high (SOTA on ImageNet) | Medium (RWKV-6) | High, growing fast | Early (research) |
| **Best for** | Text generation | Text understanding | Image classification / recognition | Long-sequence generation | Long-sequence modelling | Function approximation / scientific ML |

### § 6.2 Evolution of sequence (and image) models

```
Week 7:   RNN → LSTM → GRU → Transformer
                                  │
Week 8:   ├─── GPT   (decoder-only, generation)
          │       ↓ contrast
          ├─── BERT  (encoder-only, understanding)
          │       ↓ "apply to a new modality"
          ├─── ViT   (patchify → standard encoder stack)
          │       ↓ "but global attention is O(T²)..."
          ├─── Swin  (windowed attention, hierarchical features)
          │       ↓ "but self-attention is still O(T²) in sequence length..."
          ├─── RWKV  (linearise attention → RNN-Transformer bridge)   [App. A]
          ├─── Mamba (state space models → selective scan)
          │       ↓ "but the MLP inside every block is unchanged..."
          └─── KAN   (rethink the MLP building block itself)           [App. B]
```

The story of deep learning for sequences (and images) is one of **progressively questioning assumptions**: first the need for recurrence (Transformer), then the deployment paradigm (GPT / BERT / ViT), then the quadratic cost (Swin / RWKV / Mamba), and finally the basic MLP building block (KAN). Each generation removes an assumption the previous generation took for granted.


---

# Appendix A · RWKV (Receptance Weighted Key Value)

> **Scope note.** RWKV is **not in the Week 8 lecture**. It is included here as an *alternative route* to linear-time sequence modelling: Mamba (§ 5) takes the SSM route from control theory; RWKV takes the "linearised exponential attention with time decay" route from the attention side. Read this appendix if you want to see both routes side by side. Skip it if you just want to know what the lecture covers.

### A.1 Core Idea

RWKV bridges RNNs and Transformers by replacing softmax attention with a **linear attention** mechanism that can run in two modes:

- **Parallel** (like a Transformer) during **training**
- **Recurrent** (like an RNN, constant memory) during **inference**

This is the "best of both worlds" — Transformer-speed training with RNN-speed inference. Same meta-goal as Mamba, different mathematics.


### A.2 The WKV operator

**Standard attention** (review from Week 7):

$$\text{Attn}(q_t, K, V) = \frac{\sum_{i=1}^{T} e^{q_t \cdot k_i}\, v_i}{\sum_{i=1}^{T} e^{q_t \cdot k_i}}$$

**RWKV's reformulation** — replace the query–key dot product with a **time-decay** mechanism:

$$\text{wkv}_t = \frac{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i}\, v_i \;+\; e^{u + k_t}\, v_t}{\sum_{i=1}^{t-1} e^{-(t-1-i)w + k_i} \;+\; e^{u + k_t}}$$

where:

- $w$ is a **learned time-decay** vector — channels with large $w$ forget quickly, small $w$ remember further back (replaces positional encoding!)
- $u$ is a **learned bonus** for the current token
- $k_i, v_i$ are key and value at position $i$

#### The crucial recurrence

Define running sums:

$$a_t = e^{-w}\, a_{t-1} + e^{k_t}\, v_t, \qquad b_t = e^{-w}\, b_{t-1} + e^{k_t}$$

Then $\text{wkv}_t = a_t / b_t$. This is an **exponentially-weighted moving average** — computable as an RNN with **$O(1)$ memory** per time step.

The time-decay $w$ acts per-channel: large $w$ channels behave like short-term memory (recent tokens dominate), small $w$ channels behave like long-term memory. The "one algorithm, two views" structure parallels § 5.4's SSM duality, but from a different algebraic starting point.


### A.3 The Full RWKV Block

Each RWKV block has two sub-blocks (analogous to Transformer's attention + FFN):

**Time-mixing** (replaces attention):

- Token shift: $\bar x_t = \text{lerp}(x_{t-1}, x_t, \mu) = (1-\mu)\, x_{t-1} + \mu\, x_t$
- Receptance gate: $r_t = \sigma(W_r \bar x_t)$ — controls how much WKV output to use
- Key and Value: $k_t = W_k \bar x_t$, $v_t = W_v \bar x_t$
- Output: $o_t = W_o (r_t \odot \text{wkv}_t)$

**Channel-mixing** (replaces FFN):

- Similar structure with token-shift, but uses squared ReLU: $\text{ReLU}(x)^2$ as the activation.


In [ ]:
# ── RWKV: WKV operator and block ───────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F


def lerp(a, b, mu):
    """Linear interpolation: (1-mu)*a + mu*b."""
    return (1 - mu) * a + mu * b


class WKV(nn.Module):
    """RWKV linear attention operator — recurrent mode."""
    def __init__(self, d_model):
        super().__init__()
        self.w = nn.Parameter(torch.randn(d_model) * 0.01)  # time decay
        self.u = nn.Parameter(torch.randn(d_model) * 0.01)  # current-token bonus

    def forward(self, k, v):
        """k, v: (B, T, D) -> output: (B, T, D)"""
        B, T, D = k.shape
        output = torch.zeros_like(v)

        # Recurrent state: numerator (a) and denominator (b)
        a = torch.zeros(B, D, device=k.device)
        b = torch.zeros(B, D, device=k.device)

        decay = torch.exp(-torch.exp(self.w))  # ensure positive decay in (0,1)

        for t in range(T):
            kt, vt = k[:, t], v[:, t]
            ekt = torch.exp(kt)

            # Current token contribution (with bonus u)
            bonus = torch.exp(self.u + kt)
            wkv_t = (a + bonus * vt) / (b + bonus + 1e-8)
            output[:, t] = wkv_t

            # Update recurrent state
            a = decay * a + ekt * vt
            b = decay * b + ekt

        return output


class RWKVBlock(nn.Module):
    """Single RWKV block with time-mixing and channel-mixing."""
    def __init__(self, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        # Time-mixing parameters
        self.W_r = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.mu_r = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mu_k = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mu_v = nn.Parameter(torch.ones(d_model) * 0.5)
        self.wkv = WKV(d_model)
        # Channel-mixing parameters
        self.W_r2 = nn.Linear(d_model, d_model, bias=False)
        self.W_k2 = nn.Linear(d_model, d_model, bias=False)
        self.W_v2 = nn.Linear(d_model, d_model, bias=False)
        self.mu_r2 = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mu_k2 = nn.Parameter(torch.ones(d_model) * 0.5)

    def _token_shift(self, x):
        """Shift x right by 1 position, padding with zeros."""
        return F.pad(x[:, :-1], (0, 0, 1, 0))

    def time_mixing(self, x):
        x_prev = self._token_shift(x)
        r = torch.sigmoid(self.W_r(lerp(x_prev, x, self.mu_r)))
        k = self.W_k(lerp(x_prev, x, self.mu_k))
        v = self.W_v(lerp(x_prev, x, self.mu_v))
        return self.W_o(r * self.wkv(k, v))

    def channel_mixing(self, x):
        x_prev = self._token_shift(x)
        r = torch.sigmoid(self.W_r2(lerp(x_prev, x, self.mu_r2)))
        k = self.W_k2(lerp(x_prev, x, self.mu_k2))
        return r * self.W_v2(torch.relu(k) ** 2)  # squared ReLU

    def forward(self, x):
        """x: (B, T, D) -> (B, T, D)"""
        x = x + self.time_mixing(self.ln1(x))
        x = x + self.channel_mixing(self.ln2(x))
        return x


In [ ]:
# ── RWKV verification ───────────────────────────────────────────────────────
torch.manual_seed(42)
d_model = 64
block = RWKVBlock(d_model)
x = torch.randn(2, 20, d_model)  # batch=2, seq_len=20
out = block(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}')
print(f'Parameters:   {sum(p.numel() for p in block.parameters()):,}')


---

# Appendix B · Kolmogorov–Arnold Networks (KAN)

> **Scope note.** KAN is **not in the Week 8 lecture**, and it does not belong to the "efficient attention" thread of the rest of this notebook — it is an *orthogonal direction* that rethinks the **MLP** rather than the attention operator. Every architecture in this notebook (Transformer / BERT / GPT / ViT / Mamba / RWKV) contains standard MLP feed-forward blocks with fixed non-linearities on the *nodes*. KAN asks: what if we put **learnable** non-linearities on the **edges** instead? Included here because it was in the old tutorial notebook and makes an interesting contrast. Fully skippable.

### B.1 The Kolmogorov–Arnold representation theorem

**Theorem** (Kolmogorov 1957, Arnold 1957): any continuous function $f: [0,1]^n \to \mathbb{R}$ can be written as

$$f(x_1, \ldots, x_n) = \sum_{q=0}^{2n} \Phi_q\!\left(\sum_{p=1}^{n} \phi_{q,p}(x_p)\right)$$

where $\phi_{q,p}: [0,1] \to \mathbb{R}$ and $\Phi_q: \mathbb{R} \to \mathbb{R}$ are **continuous univariate functions**.

This is a remarkable result: **any** multivariate continuous function can be decomposed into compositions and sums of **univariate** functions. KAN (Liu et al. 2024) turns this theorem into a neural network architecture.


### B.2 MLP vs KAN — the fundamental difference

| | MLP | KAN |
| --- | --- | --- |
| **On edges** | Learnable scalar weights $w_i$ | Learnable **functions** $\phi_i(x)$ (B-splines) |
| **On nodes** | Fixed activation $\sigma$ (ReLU, etc.) | Just **summation** |
| **Neuron output** | $\text{out} = \sigma\!\left(\sum_i w_i x_i + b\right)$ | $\text{out} = \sum_i \phi_i(x_i)$ |
| **Non-linearity** | Fixed, same everywhere | Learned, unique per edge |
| **Interpretability** | Low (weights are opaque scalars) | High (can plot each $\phi_i$) |

In an MLP, non-linearity is a **fixed recipe** applied uniformly. In a KAN, non-linearity is **learned per connection** — each edge becomes a tiny function approximator.


### B.3 B-spline parameterisation

How to make each $\phi_i$ learnable? Parameterise it as a **B-spline** plus a residual:

$$\phi(x) = w_b \cdot \text{SiLU}(x) + w_s \cdot \text{spline}(x)$$

where $\text{spline}(x) = \sum_j c_j B_j(x)$, with $B_j(x)$ the B-spline basis functions of order $k$ and $c_j$ the **learnable coefficients**.

**B-spline basics** (order $k = 3$, i.e. cubic):

- Piecewise polynomials of degree $k-1$ defined on a knot vector
- **Local** — each basis function is nonzero on at most $k+1$ grid intervals
- **Smooth** — $C^{k-2}$ continuous at knots
- With $G$ grid intervals, you get $G + k$ basis functions per edge


In [ ]:
# ── KAN Layer ───────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class KANLayer(nn.Module):
    """A single KAN layer with B-spline activation functions on edges."""
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        # B-spline grid on [-1, 1] with extensions for boundary
        h = 2.0 / grid_size
        grid = torch.linspace(
            -1 - spline_order * h, 1 + spline_order * h,
            grid_size + 2 * spline_order + 1
        )
        self.register_buffer('grid', grid)

        # Learnable spline coefficients: one set per (in, out) edge
        n_basis = grid_size + spline_order
        self.spline_weight = nn.Parameter(
            torch.randn(out_features, in_features, n_basis) * 0.1
        )
        # Base weight (residual SiLU path)
        self.base_weight = nn.Parameter(
            torch.randn(out_features, in_features) *
            (2.0 / (in_features + out_features)) ** 0.5
        )

    def b_spline_basis(self, x):
        """Compute B-spline basis values via Cox-de Boor recursion.
        x: (B, in_features) -> (B, in_features, n_basis)
        """
        x = x.unsqueeze(-1)                                 # (B, in, 1)
        grid = self.grid                                     # (G + 2k + 1,)
        # Order 0: piecewise constant
        bases = ((x >= grid[:-1]) & (x < grid[1:])).float()  # (B, in, G+2k)
        # Cox-de Boor recursion for higher orders
        for k in range(1, self.spline_order + 1):
            left_num = x - grid[:-(k + 1)]
            left_den = grid[k:-1] - grid[:-(k + 1)]
            right_num = grid[k + 1:] - x
            right_den = grid[k + 1:] - grid[1:-k]
            left = left_num / (left_den + 1e-8) * bases[..., :-1]
            right = right_num / (right_den + 1e-8) * bases[..., 1:]
            bases = left + right
        return bases                                         # (B, in, n_basis)

    def forward(self, x):
        """x: (B, in_features) -> (B, out_features)"""
        # Base path: SiLU activation (residual)
        base_out = F.silu(x) @ self.base_weight.T            # (B, out)
        # Spline path: B-spline basis weighted by learnable coefficients
        spline_basis = self.b_spline_basis(x)                # (B, in, n_basis)
        spline_out = torch.einsum('bin,oin->bo',
                                   spline_basis, self.spline_weight)
        return base_out + spline_out


In [ ]:
# ── KAN verification ────────────────────────────────────────────────────────
torch.manual_seed(42)
kan_layer = KANLayer(in_features=2, out_features=5, grid_size=5, spline_order=3)
x = torch.randn(10, 2)  # batch=10, 2 input features
out = kan_layer(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}')
print(f'Spline basis shape: {kan_layer.b_spline_basis(x).shape}  (batch, in, n_basis)')
print(f'Parameters: {sum(p.numel() for p in kan_layer.parameters()):,}')


### B.4 KAN vs MLP — function-fitting comparison

Train both an MLP and a KAN on the target function $f(x, y) = \sin(\pi x) + y^2$ and compare convergence and parameter efficiency.


In [ ]:
# ── KAN vs MLP comparison on function fitting ─────────────────────────
import matplotlib.pyplot as plt

%matplotlib inline

# Target function: f(x, y) = sin(pi * x) + y^2
torch.manual_seed(42)
X_train = torch.rand(1000, 2) * 2 - 1  # (x, y) in [-1, 1]
Y_train = torch.sin(math.pi * X_train[:, 0]) + X_train[:, 1] ** 2
Y_train = Y_train.unsqueeze(-1)

X_test = torch.rand(200, 2) * 2 - 1
Y_test = torch.sin(math.pi * X_test[:, 0]) + X_test[:, 1] ** 2
Y_test = Y_test.unsqueeze(-1)

# MLP: 2 -> 32 -> 32 -> 1
mlp = nn.Sequential(
    nn.Linear(2, 32), nn.SiLU(),
    nn.Linear(32, 32), nn.SiLU(),
    nn.Linear(32, 1)
)

# KAN: 2 -> 8 -> 1
class KANNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = KANLayer(2, 8, grid_size=5)
        self.layer2 = KANLayer(8, 1, grid_size=5)
    def forward(self, x):
        return self.layer2(self.layer1(x))

kan = KANNet()

mlp_params = sum(p.numel() for p in mlp.parameters())
kan_params = sum(p.numel() for p in kan.parameters())
print(f'MLP parameters: {mlp_params:,}')
print(f'KAN parameters: {kan_params:,}')


def train_model(model, name, epochs=500):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(epochs):
        pred = model(X_train)
        loss = F.mse_loss(pred, Y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if (epoch + 1) % 100 == 0:
            with torch.no_grad():
                test_loss = F.mse_loss(model(X_test), Y_test).item()
            print(f'  [{name}] epoch {epoch+1}, train {loss.item():.4f}, test {test_loss:.4f}')
    return losses


print('\nTraining MLP...')
mlp_losses = train_model(mlp, 'MLP')
print('\nTraining KAN...')
kan_losses = train_model(kan, 'KAN')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mlp_losses, label=f'MLP ({mlp_params:,} params)', alpha=0.8)
ax.plot(kan_losses, label=f'KAN ({kan_params:,} params)', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(r'KAN vs MLP: Fitting $f(x,y) = \sin(\pi x) + y^2$')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Visualise KAN learned edge functions ───────────────────────────────
# Plot the learned activation functions on the first layer's edges.
x_vals = torch.linspace(-1, 1, 200).unsqueeze(-1)  # (200, 1)

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
fig.suptitle('KAN Layer 1: Learned Edge Functions (input -> hidden)', fontsize=13)

with torch.no_grad():
    for out_idx in range(min(8, kan.layer1.out_features)):
        ax = axes[out_idx // 4, out_idx % 4]
        for in_idx in range(2):
            basis = kan.layer1.b_spline_basis(x_vals.expand(-1, 2))
            spline_vals = (basis[:, in_idx] *
                           kan.layer1.spline_weight[out_idx, in_idx]).sum(-1)
            base_vals = F.silu(x_vals.squeeze()) * kan.layer1.base_weight[out_idx, in_idx]
            total = spline_vals + base_vals
            label = f'$x_{in_idx+1}$'
            ax.plot(x_vals.squeeze().numpy(), total.numpy(), label=label)
        ax.set_title(f'Hidden {out_idx+1}', fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Each curve shows the learned univariate function on one edge.')
print('Unlike MLP weights (opaque scalars), KAN edges are interpretable functions.')


### B.5 KAN — strengths and limitations

| | Strengths | Limitations |
| --- | --- | --- |
| **Interpretability** | Can plot each edge function | Not applicable to all domains |
| **Accuracy** | Can be more parameter-efficient for smooth functions | Not consistently better for all tasks |
| **Theory** | Grounded in the Kolmogorov–Arnold theorem | The theorem does not guarantee efficient learning |
| **Speed** | — | **Slower training** (B-spline basis computation overhead) |
| **Scale** | — | **Not yet proven at large scale** (millions of parameters) |
| **Maturity** | — | **Early research stage** — engineering not yet optimised |

**Bottom line.** KAN is an intellectually interesting alternative to the MLP that exploits a classical mathematical theorem. It is especially well-suited to **scientific machine learning** problems where interpretability matters and the target functions are smooth and low-dimensional. Whether it scales to replace MLPs in large language models or vision backbones is still an open research question.
